# Evaluate Porseman retrieval models

Evaluate the base and fine-tuned BGE-M3 models alongside Jina Embeddings v3 and Snowflake Arctic Embed L v2.0 on the same held-out Porseman test set.

## Setup

Set FINE_TUNED_MODEL_PATH to the local directory or Hugging Face model ID of the fine-tuned model. BGE models use batch size 16; Jina v3 and Snowflake Arctic use batch size 2. Each model is released from GPU memory before the next model is loaded.

In [ ]:
import csv
import gc
import json
import re
from pathlib import Path

import numpy as np
import torch
from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import SentenceTransformer

def find_project_root(start_path):
    for candidate in (start_path, *start_path.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root. Run this notebook from inside the repository.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TEST_INPUT_PATH = PROJECT_ROOT / "data/processed/porseman_test.csv"
REPORT_DIR = PROJECT_ROOT / "reports/evaluations/porseman_model_comparison"

BASE_MODEL_NAME = "BAAI/bge-m3"
FINE_TUNED_MODEL_PATH = None
JINA_MODEL_NAME = "jinaai/jina-embeddings-v3"
SNOWFLAKE_MODEL_NAME = "Snowflake/snowflake-arctic-embed-l-v2.0"

BGE_ENCODE_BATCH_SIZE = 16
COMPARISON_ENCODE_BATCH_SIZE = 2
SEARCH_BATCH_SIZE = 64
RETRIEVED_TOP_K = 5
EVALUATION_DEVICES = ["cuda:0"] if torch.cuda.is_available() else ["cpu"]
EVALUATION_DEVICE = EVALUATION_DEVICES[0]
USE_FP16 = torch.cuda.is_available()

print(f"Test input: {TEST_INPUT_PATH}")
print(f"Reports: {REPORT_DIR}")
print(f"Device: {EVALUATION_DEVICE}; FP16: {USE_FP16}")
print(f"BGE batch size: {BGE_ENCODE_BATCH_SIZE}; Jina/Snowflake batch size: {COMPARISON_ENCODE_BATCH_SIZE}")
print(f"Retrieved answers saved per query: top {RETRIEVED_TOP_K}")

## 1. Load the fixed test set

The first occurrence of an exact duplicate answer is retained so each query has one distinct relevant corpus document.

In [ ]:
REQUIRED_FIELDS = {"id", "question", "content_text"}
if not TEST_INPUT_PATH.is_file():
    raise FileNotFoundError(f"Test CSV was not found: {TEST_INPUT_PATH}")

with TEST_INPUT_PATH.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    missing_fields = REQUIRED_FIELDS - set(reader.fieldnames or [])
    if missing_fields:
        raise ValueError(f"Test CSV is missing fields: {sorted(missing_fields)}")
    raw_test_rows = [
        {field: (row[field] or "").strip() for field in REQUIRED_FIELDS}
        for row in reader
    ]

invalid_rows = [row for row in raw_test_rows if not all(row.values())]
if invalid_rows:
    raise ValueError(f"Test CSV has {len(invalid_rows):,} row(s) with an empty required field.")
if len({row["id"] for row in raw_test_rows}) != len(raw_test_rows):
    raise ValueError("Test CSV has duplicate IDs.")

seen_answers = set()
test_rows = []
duplicate_answer_rows = []
for row in raw_test_rows:
    if row["content_text"] in seen_answers:
        duplicate_answer_rows.append(row)
    else:
        seen_answers.add(row["content_text"])
        test_rows.append(row)

test_ids = [row["id"] for row in test_rows]
test_queries = [row["question"] for row in test_rows]
corpus_documents = [row["content_text"] for row in test_rows]
relevant_document_indices = np.arange(len(test_rows))

print(f"Test rows read: {len(raw_test_rows):,}")
print(f"Rows removed for duplicate answers: {len(duplicate_answer_rows):,}")
print(f"Evaluation queries and corpus documents: {len(test_rows):,}")

## 2. Define exact retrieval evaluation

Each query is compared with every corpus answer. The correct answer rank, Recall@1, Recall@5, and MRR@10 are calculated without an additional length limit.

In [ ]:
def l2_normalize(embeddings):
    embeddings = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("The model produced a zero-length embedding.")
    return embeddings / norms

def safe_name(value):
    name = str(value).rstrip("/\\").replace("\\", "/").split("/")[-1]
    return re.sub(r"[^A-Za-z0-9._-]+", "-", name).strip("-_") or "model"

def calculate_retrieval_results(query_embeddings, corpus_embeddings, relevant_indices):
    normalized_queries = l2_normalize(query_embeddings)
    normalized_corpus = l2_normalize(corpus_embeddings)
    corpus_transposed = np.ascontiguousarray(normalized_corpus.T)
    ranks = np.empty(len(normalized_queries), dtype=np.int64)
    top_k = min(RETRIEVED_TOP_K, len(normalized_corpus))
    top_indices = np.empty((len(normalized_queries), top_k), dtype=np.int64)
    top_scores = np.empty((len(normalized_queries), top_k), dtype=np.float32)

    for start in range(0, len(normalized_queries), SEARCH_BATCH_SIZE):
        end = min(start + SEARCH_BATCH_SIZE, len(normalized_queries))
        scores = normalized_queries[start:end] @ corpus_transposed
        batch_indices = np.arange(end - start)
        positive_scores = scores[batch_indices, relevant_indices[start:end]]
        ranks[start:end] = (scores > positive_scores[:, None]).sum(axis=1) + 1
        candidate_indices = np.argpartition(-scores, top_k - 1, axis=1)[:, :top_k]
        candidate_scores = np.take_along_axis(scores, candidate_indices, axis=1)
        order = np.argsort(-candidate_scores, axis=1, kind='stable')
        top_indices[start:end] = np.take_along_axis(candidate_indices, order, axis=1)
        top_scores[start:end] = np.take_along_axis(candidate_scores, order, axis=1)

    return ranks, top_indices, top_scores

def release_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, "ipc_collect"):
            torch.cuda.ipc_collect()

def encode_bge(model_name_or_path, batch_size):
    model = None
    try:
        model = BGEM3FlagModel(
            model_name_or_path,
            use_fp16=USE_FP16,
            pooling_method="cls",
            devices=EVALUATION_DEVICES,
        )
        query_embeddings = model.encode_queries(
            test_queries, batch_size=batch_size, return_dense=True,
            return_sparse=False, return_colbert_vecs=False,
        )["dense_vecs"]
        corpus_embeddings = model.encode_corpus(
            corpus_documents, batch_size=batch_size, return_dense=True,
            return_sparse=False, return_colbert_vecs=False,
        )["dense_vecs"]
        return query_embeddings, corpus_embeddings
    finally:
        del model
        release_gpu_memory()

def encode_sentence_transformer(model_name_or_path, batch_size, query_kwargs, corpus_kwargs, trust_remote_code=False):
    model = None
    try:
        model = SentenceTransformer(
            model_name_or_path, trust_remote_code=trust_remote_code, device=EVALUATION_DEVICE
        )
        query_embeddings = model.encode(
            test_queries, batch_size=batch_size, convert_to_numpy=True,
            show_progress_bar=True, **query_kwargs,
        )
        corpus_embeddings = model.encode(
            corpus_documents, batch_size=batch_size, convert_to_numpy=True,
            show_progress_bar=True, **corpus_kwargs,
        )
        return query_embeddings, corpus_embeddings
    finally:
        del model
        release_gpu_memory()

def evaluate_model(model_spec):
    family = model_spec["family"]
    if family == "bge":
        query_embeddings, corpus_embeddings = encode_bge(
            model_spec["model"], model_spec["batch_size"]
        )
    elif family == "jina_v3":
        query_embeddings, corpus_embeddings = encode_sentence_transformer(
            model_spec["model"], model_spec["batch_size"],
            query_kwargs={"task": "retrieval.query", "prompt_name": "retrieval.query"},
            corpus_kwargs={"task": "retrieval.passage", "prompt_name": "retrieval.passage"},
            trust_remote_code=True,
        )
    elif family == "snowflake_arctic":
        query_embeddings, corpus_embeddings = encode_sentence_transformer(
            model_spec["model"], model_spec["batch_size"],
            query_kwargs={"prompt_name": "query"}, corpus_kwargs={},
        )
    else:
        raise ValueError(f"Unsupported model family: {family}")

    try:
        correct_ranks, top_indices, top_scores = calculate_retrieval_results(
            query_embeddings, corpus_embeddings, relevant_document_indices
        )
        metrics = {
            "recall_at_1": float(np.mean(correct_ranks <= 1)),
            "recall_at_5": float(np.mean(correct_ranks <= 5)),
            "mrr_at_10": float(np.mean(np.where(correct_ranks <= 10, 1.0 / correct_ranks, 0.0))),
        }
        return {
            "label": model_spec["label"], "model": model_spec["model"],
            "metrics": metrics, "correct_ranks": correct_ranks,
            "top_indices": top_indices, "top_scores": top_scores,
            "settings": model_spec["settings"],
        }
    finally:
        del query_embeddings, corpus_embeddings
        release_gpu_memory()


## 3. Evaluate four retrieval models

All models use the same test rows, corpus, dense cosine-similarity search, and metrics. Each model uses its official query/passage encoding convention and is released from GPU memory before the next model starts.

In [ ]:
models_to_evaluate = [
    {
        "label": "base_bge_m3", "model": BASE_MODEL_NAME, "family": "bge",
        "batch_size": BGE_ENCODE_BATCH_SIZE,
        "settings": {"encoder": "BGEM3FlagModel", "pooling": "cls", "query_mode": "encode_queries", "corpus_mode": "encode_corpus"},
    },
    {
        "label": "jina_embeddings_v3", "model": JINA_MODEL_NAME, "family": "jina_v3",
        "batch_size": COMPARISON_ENCODE_BATCH_SIZE,
        "settings": {"encoder": "SentenceTransformer", "query_task": "retrieval.query", "corpus_task": "retrieval.passage"},
    },
    {
        "label": "snowflake_arctic_embed_l_v2", "model": SNOWFLAKE_MODEL_NAME, "family": "snowflake_arctic",
        "batch_size": COMPARISON_ENCODE_BATCH_SIZE,
        "settings": {"encoder": "SentenceTransformer", "query_prompt_name": "query", "corpus_prompt_name": None},
    },
]
if FINE_TUNED_MODEL_PATH:
    models_to_evaluate.insert(1, {
        "label": "fine_tuned_bge_m3", "model": FINE_TUNED_MODEL_PATH, "family": "bge",
        "batch_size": BGE_ENCODE_BATCH_SIZE,
        "settings": {"encoder": "BGEM3FlagModel", "pooling": "cls", "query_mode": "encode_queries", "corpus_mode": "encode_corpus"},
    })

evaluation_results = []
for model_spec in models_to_evaluate:
    print(f"Evaluating {model_spec['label']}: {model_spec['model']} (batch size {model_spec['batch_size']})")
    result = evaluate_model(model_spec)
    evaluation_results.append(result)
    print(
        f"{result['label']} - Recall@1: {result['metrics']['recall_at_1']:.2%}; "
        f"Recall@5: {result['metrics']['recall_at_5']:.2%}; "
        f"MRR@10: {result['metrics']['mrr_at_10']:.2%}"
    )

## 4. Save comparable metrics and per-query ranks

Every run writes one metrics JSON and one rank CSV per model, plus a single comparison CSV.

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
comparison_rows = []

for result in evaluation_results:
    label = safe_name(result["label"])
    metric_path = REPORT_DIR / f"metrics_{label}.json"
    rank_path = REPORT_DIR / f"per_query_ranks_{label}.csv"
    retrieval_path = REPORT_DIR / f"per_query_top_{RETRIEVED_TOP_K}_{label}.jsonl"
    report = {
        "model": result["model"],
        "test_file": str(TEST_INPUT_PATH.resolve()),
        "query_count": len(test_rows),
        "corpus_document_count": len(corpus_documents),
        "metrics": result["metrics"],
        "settings": {
            **result["settings"],
            "devices": EVALUATION_DEVICES,
            "fp16": USE_FP16,
            "encode_batch_size": next(spec["batch_size"] for spec in models_to_evaluate if spec["label"] == result["label"]),
            "search_batch_size": SEARCH_BATCH_SIZE,
        },
    }
    metric_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    with rank_path.open("w", encoding="utf-8-sig", newline="") as target:
        writer = csv.DictWriter(target, fieldnames=["id", "question", "correct_answer_rank", "recall_at_1", "recall_at_5", "reciprocal_rank_at_10"])
        writer.writeheader()
        writer.writerows({
            "id": row_id,
            "question": question,
            "correct_answer_rank": int(rank),
            "recall_at_1": int(rank <= 1),
            "recall_at_5": int(rank <= 5),
            "reciprocal_rank_at_10": float(1.0 / rank) if rank <= 10 else 0.0,
        } for row_id, question, rank in zip(test_ids, test_queries, result["correct_ranks"]))

    with retrieval_path.open("w", encoding="utf-8") as target:
        for row_id, question, rank, document_indices, scores in zip(
            test_ids, test_queries, result["correct_ranks"], result["top_indices"], result["top_scores"]
        ):
            retrieved = [
                {"rank": position, "answer_id": test_ids[int(document_index)], "score": float(score)}
                for position, (document_index, score) in enumerate(zip(document_indices, scores), 1)
            ]
            target.write(json.dumps({
                "id": row_id, "question": question, "correct_answer_id": row_id,
                "correct_answer_rank": int(rank), "retrieved": retrieved,
            }, ensure_ascii=False) + "\n")

    comparison_rows.append({"label": result["label"], "model": result["model"], **result["metrics"]})

comparison_path = REPORT_DIR / "metrics_comparison.csv"
with comparison_path.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=["label", "model", "recall_at_1", "recall_at_5", "mrr_at_10"])
    writer.writeheader()
    writer.writerows(comparison_rows)

for row in comparison_rows:
    print(
        f"{row['label']}: Recall@1={row['recall_at_1']:.2%}, "
        f"Recall@5={row['recall_at_5']:.2%}, MRR@10={row['mrr_at_10']:.2%}"
    )
print(f"Comparison: {comparison_path}")

## 5. Create an HTML evaluation report

Create a self-contained report for comparing the base model, the fine-tuned model, Jina Embeddings v3, and Snowflake Arctic Embed L v2.0.

In [ ]:
import html
from datetime import datetime

metric_labels = {
    "recall_at_1": "Recall@1",
    "recall_at_5": "Recall@5",
    "mrr_at_10": "MRR@10",
}
metric_cards = "".join(
    f"<article class='metric-card'><span>{html.escape(row['label'])} - {metric_labels[metric]}</span><strong>{row[metric]:.2%}</strong><small>{row[metric]:.6f}</small></article>"
    for row in comparison_rows
    for metric in metric_labels
)
comparison_table_rows = "".join(
    f"<tr><td>{html.escape(row['label'])}</td><td>{html.escape(row['model'])}</td><td>{row['recall_at_1']:.2%}</td><td>{row['recall_at_5']:.2%}</td><td>{row['mrr_at_10']:.2%}</td></tr>"
    for row in comparison_rows
)

html_report = """<!doctype html>
<html lang='en'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>Porseman Retrieval Evaluation</title>
  <style>
    :root { --ink:#18212b; --muted:#667085; --line:#d7dee6; --paper:#ffffff; --canvas:#f3f6f8; --teal:#087f73; --teal-soft:#e7f5f2; --orange:#b54708; }
    * { box-sizing:border-box; }
    body { margin:0; background:var(--canvas); color:var(--ink); font:15px/1.55 Inter,Segoe UI,Arial,sans-serif; }
    main { width:min(1120px,calc(100% - 40px)); margin:36px auto 48px; }
    header { border-bottom:4px solid var(--teal); background:var(--paper); padding:32px 36px; }
    .eyebrow { color:var(--teal); font-size:12px; font-weight:700; letter-spacing:.08em; text-transform:uppercase; }
    h1 { margin:8px 0; font-size:32px; line-height:1.15; letter-spacing:0; }
    .subtitle { margin:0; color:var(--muted); max-width:720px; }
    .facts { display:grid; grid-template-columns:repeat(3,1fr); border:1px solid var(--line); border-top:0; background:var(--paper); }
    .fact { min-height:88px; padding:18px 22px; border-right:1px solid var(--line); }
    .fact:last-child { border:0; }
    .fact span { display:block; color:var(--muted); font-size:12px; }
    .fact strong { display:block; margin-top:4px; overflow-wrap:anywhere; }
    section { margin-top:24px; background:var(--paper); border:1px solid var(--line); padding:28px; }
    h2 { margin:0 0 18px; font-size:19px; letter-spacing:0; }
    .metrics { display:grid; grid-template-columns:repeat(3,1fr); gap:12px; }
    .metric-card { border-left:4px solid var(--teal); background:var(--teal-soft); padding:16px; min-height:118px; }
    .metric-card span, .metric-card small { display:block; color:var(--muted); font-size:12px; }
    .metric-card strong { display:block; margin:6px 0; color:var(--teal); font-size:27px; }
    table { width:100%; border-collapse:collapse; }
    th, td { padding:12px 10px; border-bottom:1px solid var(--line); text-align:left; vertical-align:top; }
    th { color:var(--muted); font-size:12px; font-weight:700; }
    td:nth-child(n+3) { color:var(--teal); font-weight:700; white-space:nowrap; }
    .note { color:var(--muted); margin:0; }
    footer { padding:18px 4px; color:var(--muted); font-size:12px; }
    @media (max-width:760px) { main { width:min(100% - 24px,1120px); margin-top:12px; } header, section { padding:22px; } .facts, .metrics { grid-template-columns:1fr; } .fact { border-right:0; border-bottom:1px solid var(--line); } .fact:last-child { border-bottom:0; } table { font-size:13px; } }
  </style>
</head>
<body>
  <main>
    <header>
      <div class='eyebrow'>Dense retrieval benchmark</div>
      <h1>Porseman Model Evaluation</h1>
      <p class='subtitle'>BGE-M3 baseline, fine-tuned BGE-M3, Jina Embeddings v3, and Snowflake Arctic Embed L v2.0 evaluated on the same held-out test set.</p>
    </header>
    <div class='facts'>
      <div class='fact'><span>Generated</span><strong>__GENERATED_AT__</strong></div>
      <div class='fact'><span>Queries / Corpus</span><strong>__QUERY_COUNT__ / __CORPUS_COUNT__</strong></div>
      <div class='fact'><span>Device</span><strong>__DEVICE__</strong></div>
    </div>
    <section><h2>Metrics</h2><div class='metrics'>__METRIC_CARDS__</div></section>
    <section>
      <h2>Model comparison</h2>
      <table><thead><tr><th>Label</th><th>Model</th><th>Recall@1</th><th>Recall@5</th><th>MRR@10</th></tr></thead><tbody>__COMPARISON_ROWS__</tbody></table>
    </section>
    <section><h2>Reproducibility</h2><p class='note'>Test file: __TEST_FILE__<br>FP16: __FP16__ | BGE encode batch size: __BGE_ENCODE_BATCH_SIZE__ | Jina/Snowflake encode batch size: __COMPARISON_ENCODE_BATCH_SIZE__ | Search batch size: __SEARCH_BATCH_SIZE__</p></section>
    <footer>Generated by the Porseman retrieval evaluation notebook.</footer>
  </main>
</body>
</html>
"""
html_report = (html_report
    .replace("__GENERATED_AT__", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z"))
    .replace("__QUERY_COUNT__", f"{len(test_rows):,}")
    .replace("__CORPUS_COUNT__", f"{len(corpus_documents):,}")
    .replace("__DEVICE__", html.escape(EVALUATION_DEVICES[0]))
    .replace("__METRIC_CARDS__", metric_cards)
    .replace("__COMPARISON_ROWS__", comparison_table_rows)
    .replace("__TEST_FILE__", html.escape(str(TEST_INPUT_PATH.resolve())))
    .replace("__FP16__", str(USE_FP16))
    .replace("__BGE_ENCODE_BATCH_SIZE__", str(BGE_ENCODE_BATCH_SIZE))
    .replace("__COMPARISON_ENCODE_BATCH_SIZE__", str(COMPARISON_ENCODE_BATCH_SIZE))
    .replace("__SEARCH_BATCH_SIZE__", str(SEARCH_BATCH_SIZE))
)
html_report_path = REPORT_DIR / "evaluation_report.html"
html_report_path.write_text(html_report, encoding="utf-8")
print(f"HTML report: {html_report_path}")